# Week 5: Prompt Library on GSM8K

<a href="https://colab.research.google.com/github/jorshimayor/AI-Engineering-Flow_Research/blob/main/jupyter/week5_prompt_library_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Five prompting strategies ([`src/prompts.py`](../src/prompts.py)) on 400 GSM8K test problems, with Qwen2.5-0.5B-Instruct and 1.5B-Instruct. The plan and hypotheses were committed before this run: [`notes/week_5/README.md`](../notes/week_5/README.md).

**Run:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`. About 40 minutes. Results save to Google Drive as each batch finishes, so re-running after a disconnect continues where it stopped.

## 1. GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → T4 GPU"
print(torch.cuda.get_device_name(0))

## 2. Code (always the latest)

In [ ]:
REPO = "https://github.com/jorshimayor/AI-Engineering-Flow_Research.git"
BRANCH = "main"
import os
BASE = "/content" if os.path.isdir("/content") else os.getcwd()
CODE = os.path.join(BASE, "AI-Engineering-Flow_Research")
if not os.path.exists(CODE):
    !git clone --depth 1 --branch {BRANCH} {REPO} {CODE}
%cd {CODE}
!git fetch -q --depth 1 origin {BRANCH} && git checkout -q -B {BRANCH} FETCH_HEAD
!pip install -q "transformers==4.49.0" accelerate
!git log --oneline -1

## 3. Results on Google Drive

In [ ]:
USE_DRIVE = True
RESULTS = "results/week5"
if USE_DRIVE:
    import shutil
    from google.colab import drive
    drive.mount("/content/drive")
    target = "/content/drive/MyDrive/AI-Engineering/week5"
    os.makedirs(target, exist_ok=True)
    if os.path.isdir(RESULTS) and not os.path.islink(RESULTS):   # keep files the repo already has here
        for f in os.listdir(RESULTS):
            if os.path.isfile(os.path.join(RESULTS, f)) and not os.path.exists(os.path.join(target, f)):
                shutil.copy(os.path.join(RESULTS, f), target)
        shutil.rmtree(RESULTS)
    os.makedirs(os.path.dirname(RESULTS), exist_ok=True)
    if not os.path.islink(RESULTS):
        os.symlink(target, RESULTS)
os.makedirs(RESULTS, exist_ok=True)
print("results go to:", os.path.realpath(RESULTS))

## 4. The prompt library

In [ ]:
import sys, importlib
sys.path.insert(0, "src")
import prompts; importlib.reload(prompts)
from prompts import STRATEGIES, LIBRARY_VERSION, BASELINE
from gsm8k import sample
assert LIBRARY_VERSION == "1.1", f"library {LIBRARY_VERSION}: Runtime → Disconnect and delete runtime, then Run all"
print("library version", LIBRARY_VERSION, "| baseline:", BASELINE)
for s in STRATEGIES.values():
    print(f"  {s.name:18} {s.description}")
PROBLEMS = sample(400)
print(len(PROBLEMS), "GSM8K test problems")

## 5. Run

Both models, all five strategies. Each model is loaded once; generation is batched.

In [ ]:
import gc
from pathlib import Path
from run_prompt_bench import BatchLLM, run

MODELS = ["Qwen/Qwen2.5-0.5B-Instruct", "Qwen/Qwen2.5-1.5B-Instruct"]
BATCH_SIZE = 16

for model in MODELS:
    llm = BatchLLM(model)
    run(model, list(STRATEGIES), 400, Path(RESULTS) / f"{model.split('/')[-1]}.jsonl", BATCH_SIZE, llm=llm)
    del llm; gc.collect(); torch.cuda.empty_cache()

## 6. Results

In [ ]:
from run_prompt_bench import summary
summary(sorted(Path(RESULTS).glob("*.jsonl")))

## 7. Figures

In [ ]:
import figures
importlib.reload(figures)
figures.OUT = Path(RESULTS) / "figures"
figures.RES = Path("results")
figures.week5_strategy_accuracy()
figures.week5_cost_vs_accuracy()
from IPython.display import Image, display
for f in sorted(figures.OUT.glob("week5_*.png")):
    display(Image(str(f)))

## 8. Download

In [ ]:
zip_path = os.path.join(BASE, "week5_results.zip")
!cd results && zip -qr {zip_path} week5/
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Download", zip_path, "from the output panel")

Send the zip back, or unzip it into `results/` in the repo.